# Experiment 10: Building a Supervised Noun-Phrase Chunker

**Mapped CO:** CO5  
**BT Level:** BT5

## Aim
Build a supervised noun-phrase chunker using contextual token features and a statistical classifier.

## Objectives
- Convert chunk trees into sequence labels for supervised learning.
- Design lexical, POS, and context features for chunk decisions.
- Train a classifier-based sequence tagger.
- Evaluate a learned chunker on unseen annotated sentences.

## Theory
A learned chunker predicts chunk labels from annotated examples rather than relying only on hand-written patterns. Each token is represented by features such as its word, POS tag, neighboring POS tags, previous word, and previous predicted IOB label. A classifier predicts the IOB label sequence one token at a time.

The CoNLL-2000 corpus contains POS-tagged text with chunk annotations. This experiment keeps only noun-phrase (`NP`) chunks and trains an NLTK classifier-based tagger on a configurable subset so that classroom execution remains practical.

## Methodology
1. Load NP-only chunk trees from the CoNLL-2000 corpus.
2. Convert each tree to word/POS/IOB sequences.
3. Define lexical and contextual token features.
4. Train a classifier-based sequence tagger.
5. Wrap the tagger as an NLTK `ChunkParserI` component.
6. Evaluate on held-out chunk trees and inspect a sample parse.

## Dataset and requirements
- Python 3.10 or above
- `nltk`
- NLTK CoNLL-2000 corpus
- Notebook or Colab

The training and test limits below are deliberately exposed as variables. Increase them when more time and memory are available.

In [10]:
import random
import nltk
from nltk.chunk import ChunkParserI
from nltk.chunk.util import ChunkScore, conlltags2tree, tree2conlltags
from nltk.classify import MaxentClassifier
from nltk.tag import ClassifierBasedTagger

# Download only the resources needed by this notebook when they are absent.
resources = {
    "tokenizers/punkt_tab": "punkt_tab",
    "taggers/averaged_perceptron_tagger_eng": "averaged_perceptron_tagger_eng",
    "corpora/conll2000/train": "conll2000",
}
for resource_path, resource_name in resources.items():
    try:
        nltk.data.find(resource_path)
    except LookupError:
        nltk.download(resource_name, quiet=True)

SEED = 7
TRAIN_LIMIT = 1000
TEST_LIMIT = 200
random.seed(SEED)
print(f"NLTK version: {nltk.__version__}")
print(f"Configuration: train={TRAIN_LIMIT}, test={TEST_LIMIT}, seed={SEED}")

NLTK version: 3.10.4
Configuration: train=1000, test=200, seed=7


In [11]:
from nltk.corpus import conll2000

# NP-only trees make the target task noun-phrase chunking.
all_train_trees = conll2000.chunked_sents("train.txt", chunk_types=["NP"])
all_test_trees = conll2000.chunked_sents("test.txt", chunk_types=["NP"])

train_trees = list(all_train_trees[:TRAIN_LIMIT])
test_trees = list(all_test_trees[:TEST_LIMIT])

if not train_trees or not test_trees:
    raise RuntimeError("The CoNLL-2000 corpus did not provide train and test trees.")

def tree_to_iob(tree):
    """Return (word, POS, IOB) triples for one chunk tree."""
    return tree2conlltags(tree)

train_sequences = [tree_to_iob(tree) for tree in train_trees]
test_sequences = [tree_to_iob(tree) for tree in test_trees]
print(f"Loaded {len(train_sequences)} training and {len(test_sequences)} test trees.")
print("First training sequence:", train_sequences[0][:5])

Loaded 1000 training and 200 test trees.
First training sequence: [('Confidence', 'NN', 'B-NP'), ('in', 'IN', 'O'), ('the', 'DT', 'B-NP'), ('pound', 'NN', 'I-NP'), ('is', 'VBZ', 'O')]


## Contextual feature design

The classifier receives the current word and POS tag, a small lexical context, neighboring POS tags, and the previous predicted IOB label. The previous label helps the model learn valid continuation patterns such as `B-NP` followed by `I-NP`.

In [12]:
def token_features(tokens, index, history):
    """Features for a token sequence of (word, POS) pairs."""
    word, pos = tokens[index]
    previous_word, previous_pos = ("<START>", "<START>") if index == 0 else tokens[index - 1]
    next_word, next_pos = ("<END>", "<END>") if index == len(tokens) - 1 else tokens[index + 1]
    previous_chunk = "<START>" if not history else history[-1]

    return {
        "word": word,
        "word.lower": word.lower(),
        "word.isupper": word.isupper(),
        "word.isdigit": word.isdigit(),
        "pos": pos,
        "previous_word": previous_word,
        "previous_pos": previous_pos,
        "next_word": next_word,
        "next_pos": next_pos,
        "previous_chunk": previous_chunk,
    }

In [13]:
def sequence_training_examples(sequences):
    """Convert IOB sequences to (tokens, labels) examples."""
    examples = []
    for sequence in sequences:
        tokens = [(word, pos) for word, pos, _ in sequence]
        labels = [iob for _, _, iob in sequence]
        examples.append(list(zip(tokens, labels)))
    return examples

training_examples = sequence_training_examples(train_sequences)
print("Example tokens:", [token for token, _ in training_examples[0][:5]])
print("Example labels:", [label for _, label in training_examples[0][:5]])

Example tokens: [('Confidence', 'NN'), ('in', 'IN'), ('the', 'DT'), ('pound', 'NN'), ('is', 'VBZ')]
Example labels: ['B-NP', 'O', 'B-NP', 'I-NP', 'O']


## Train the classifier-based sequence tagger

NLTK's iterative scaling (`iis`) MaxEnt algorithm is self-contained and avoids requiring an external `megam` executable. If the local NLTK installation cannot provide it, the cell raises an explicit error instead of silently producing a fake result.

In [14]:
def maxent_classifier_builder(train_data, **kwargs):
    try:
        return MaxentClassifier.train(
            train_data,
            algorithm="iis",
            trace=0,
            max_iter=10,
        )
    except (LookupError, ValueError) as error:
        raise RuntimeError(
            "NLTK could not train the MaxEnt classifier with the built-in "
            "'iis' algorithm. Check the NLTK installation and try again."
        ) from error

tagger = ClassifierBasedTagger(
    train=training_examples,
    feature_detector=token_features,
    classifier_builder=maxent_classifier_builder,
)
print("Classifier-based sequence tagger trained successfully.")

Classifier-based sequence tagger trained successfully.


In [15]:
class SupervisedNPChunker(ChunkParserI):
    """An NLTK chunk parser backed by the trained IOB tagger."""

    def __init__(self, sequence_tagger):
        self.tagger = sequence_tagger

    def parse(self, tagged_sentence):
        tagged_with_iob = self.tagger.tag(tagged_sentence)
        conll_tags = [
            (word, pos, iob)
            for (word, pos), iob in tagged_with_iob
        ]
        return conlltags2tree(conll_tags)

chunker = SupervisedNPChunker(tagger)
print(chunker)

## Evaluation on unseen annotated sentences

Chunk precision is the proportion of predicted NP chunks that are correct. Recall is the proportion of gold NP chunks that were found. F-measure is the harmonic mean of precision and recall.

In [16]:
score = ChunkScore()
for gold_tree in test_trees:
    predicted_tree = chunker.parse(gold_tree.leaves())
    score.score(gold_tree, predicted_tree)

print(f"NP chunk precision: {score.precision():.3f}")
print(f"NP chunk recall:    {score.recall():.3f}")
print(f"NP chunk F1-score:   {score.f_measure():.3f}")
print(f"Evaluated on {len(test_trees)} held-out sentences.")

NP chunk precision: 0.851
NP chunk recall:    0.887
NP chunk F1-score:   0.869
Evaluated on 200 held-out sentences.


In [17]:
sample_text = "The small research team developed a reliable language model."
sample_tagged = nltk.pos_tag(nltk.word_tokenize(sample_text))
sample_tree = chunker.parse(sample_tagged)

print("POS-tagged input:")
print(sample_tagged)
print("\nPredicted NP chunk tree:")
print(sample_tree.pformat())
print(sample_tree.pformat())

POS-tagged input:
[('The', 'DT'), ('small', 'JJ'), ('research', 'NN'), ('team', 'NN'), ('developed', 'VBD'), ('a', 'DT'), ('reliable', 'JJ'), ('language', 'NN'), ('model', 'NN'), ('.', '.')]

Predicted NP chunk tree:
(S
  (NP The/DT small/JJ research/NN team/NN)
  developed/VBD
  (NP a/DT reliable/JJ language/NN model/NN)
  ./.)
(S
  (NP The/DT small/JJ research/NN team/NN)
  developed/VBD
  (NP a/DT reliable/JJ language/NN model/NN)
  ./.)


## Results and interpretation

The printed report gives the measured precision, recall, and F1 for the selected held-out subset. The exact values depend on the training limit, MaxEnt iterations, NLTK version, and random seed; they should be reported from the executed output rather than hardcoded.

A learned chunker can outperform fixed rules on varied text because it learns distributional patterns from annotated examples. It can be improved by increasing the training set, adding word prefixes and suffixes, adding richer POS context, tuning classifier iterations, using a stronger classifier, and checking errors by chunk type and sentence domain.

## Outcomes
- Built a supervised statistical chunker.
- Designed contextual features for sequence labels.
- Evaluated chunking with precision, recall, and F-measure.
- Explained why learned chunkers can outperform fixed rules on varied text.

## Resources
- [NLTK Chapter 7: Extracting Information from Text](https://www.nltk.org/book/ch07.html)
- [NLTK chunk API](https://www.nltk.org/api/nltk.chunk.html)
- [Speech and Language Processing](https://web.stanford.edu/~jurafsky/slp3/)
- [Demonstrative video search](https://www.youtube.com/results?search_query=build+chunker+classifier+IOB+CoNLL+NLP)

## Self-assessment questions
1. Why build a learned chunker instead of only using rules?
2. What information is present in the CoNLL-2000 chunking corpus?
3. Why are IOB labels suitable for chunking?
4. Give three useful features for a chunk classifier.
5. Why can the previous predicted IOB tag be useful?
6. What is sequence labeling?
7. Define precision for chunk evaluation.
8. Define recall for chunk evaluation.
9. What is F1-score?
10. How can the chunker be improved beyond the baseline experiment?